# Fine-tune ResNet-50 on all VisA products
Train the last ResNet block and classifier. Compare two backbone learning rates on two stratified validation splits. Each run starts from identical pretrained weights and classifier initialization. Select the learning rate by mean best validation average precision across splits. Use the first split's selected checkpoint and threshold for test evaluation.

Run from top to bottom on BigMac. This uses one GPU and processes images each epoch. Existing downloaded images are reused. Results go to a new folder, preserving previous frozen-feature results. The test set has already informed earlier experiments, so this remains exploratory.


## 1. Imports and settings

In [1]:
import csv
import math
import shutil
from torch.utils.data import Subset
from sklearn.metrics import confusion_matrix
import subprocess
import tarfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import ResNet50_Weights, resnet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, RocCurveDisplay,
)

torch.set_num_threads(4)
SEED = 42
SPLIT_SEEDS = [42, 123]
GPU_INDEX = 0
BATCH_SIZE = 16
EPOCHS = 10
BACKBONE_RATES = [1e-5, 1e-4]
HEAD_RATE = 1e-3
VALIDATION_FRACTION = 0.2
TARGET_RECALL = 0.9
DATA_DIR = Path("visa_data")
OUTPUT_DIR = Path("results/full_visa_resnet50_finetuned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device(f"cuda:{GPU_INDEX}" if torch.cuda.is_available() else "cpu")
print("Using", device)


Using cuda:0


## 2. Load the data
Reuse the official VisA split and extract missing images if necessary.

In [2]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download(url, destination):
    if destination.is_file():
        return
    partial = destination.with_suffix(destination.suffix + ".part")
    subprocess.run([
        "curl", "--fail", "--location", "--retry", "2",
        "--output", str(partial), url,
    ], check=True)
    partial.replace(destination)

split_file = DATA_DIR / "2cls_highshot.csv"
download("https://raw.githubusercontent.com/amazon-science/spot-diff/main/split_csv/2cls_highshot.csv", split_file)
with split_file.open(newline="") as file:
    rows = list(csv.DictReader(file))
assert rows, "No images found for the selected product."

missing_images = {row["image"] for row in rows if not (DATA_DIR / row["image"]).is_file()}
if missing_images:
    archive = DATA_DIR / "VisA_20220922.tar"
    download("https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar", archive)
    with tarfile.open(archive) as tar:
        for member in tar:
            relative_path = Path(member.name)
            if str(relative_path) in missing_images and member.isfile():
                destination = (DATA_DIR / relative_path).resolve()
                assert DATA_DIR.resolve() in destination.parents
                destination.parent.mkdir(parents=True, exist_ok=True)
                with tar.extractfile(member) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
assert all((DATA_DIR / row["image"]).is_file() for row in rows), "Some images are missing."
table = pd.DataFrame(rows)
display(pd.crosstab([table["object"], table["split"]], table["label"]))


label             anomaly  normal
object     split                 
candle     test        40     400
           train       60     600
capsules   test        40     241
           train       60     361
cashew     test        40     200
           train       60     300
chewinggum test        40     201
           train       60     302
fryum      test        40     200
           train       60     300
macaroni1  test        40     400
           train       60     600
macaroni2  test        40     400
           train       60     600
pcb1       test        40     402
           train       60     602
pcb2       test        40     400
           train       60     601
pcb3       test        40     402
           train       60     604
pcb4       test        40     402
           train       60     603
pipe_fryum test        40     200
           train       60     300

## 3. Prepare images and the model
Freeze earlier layers and train layer4 plus the final layer. Keep batch normalization statistics fixed because batches are small.

In [3]:
weights = ResNet50_Weights.DEFAULT
preprocess = weights.transforms()

class VisaDataset(Dataset):
    def __init__(self, split):
        self.examples = [row for row in rows if row["split"] == split]

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        row = self.examples[index]
        with Image.open(DATA_DIR / row["image"]) as image:
            pixels = preprocess(image.convert("RGB"))
        label = torch.tensor({"normal": 0, "anomaly": 1}[row["label"]], dtype=torch.float32)
        return pixels, label

train_data = VisaDataset("train")
test_data = VisaDataset("test")
assert len(train_data) and len(test_data)
assert {r["image"] for r in train_data.examples}.isdisjoint(r["image"] for r in test_data.examples)

def build_model():
    torch.manual_seed(SEED)
    model = resnet50(weights=weights)
    model.requires_grad_(False)
    model.layer4.requires_grad_(True)
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model.to(device)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    scores, targets = [], []
    for images, labels in loader:
        logits = model(images.to(device)).squeeze(1)
        scores.append(logits.sigmoid().cpu())
        targets.append(labels)
    return torch.cat(scores).numpy(), torch.cat(targets).numpy().astype(int)

def choose_threshold(labels, scores):
    defect_scores = sorted(scores[labels == 1], reverse=True)
    required = math.ceil(TARGET_RECALL * len(defect_scores))
    return float(defect_scores[required - 1])


## 4. Compare learning rates on two validation splits
Run four experiments. Save the epoch with the highest validation average precision for each run. Loss is unweighted to match the selected frozen-feature baseline. Thresholds use validation data only.

In [ ]:
strata = [row["object"] + "_" + row["label"] for row in train_data.examples]
run_rows, history_rows = [], []
for split_seed in SPLIT_SEEDS:
    fit_indices, validation_indices = train_test_split(
        range(len(train_data)), test_size=VALIDATION_FRACTION,
        stratify=strata, random_state=split_seed,
    )
    validation_loader = DataLoader(Subset(train_data, validation_indices), batch_size=BATCH_SIZE)
    for backbone_rate in BACKBONE_RATES:
        generator = torch.Generator().manual_seed(SEED)
        fit_loader = DataLoader(
            Subset(train_data, fit_indices), batch_size=BATCH_SIZE,
            shuffle=True, generator=generator,
        )
        model = build_model()
        optimizer = torch.optim.AdamW([
            {"params": model.layer4.parameters(), "lr": backbone_rate},
            {"params": model.fc.parameters(), "lr": HEAD_RATE},
        ], weight_decay=1e-4)
        loss_function = nn.BCEWithLogitsLoss()
        best_ap = -1.0
        checkpoint = OUTPUT_DIR / f"split_{split_seed}_lr_{backbone_rate:g}.pt"
        for epoch in range(1, EPOCHS + 1):
            # Evaluation mode freezes batch normalization, but gradients still update weights.
            model.eval()
            total_loss = 0.0
            for images, labels in fit_loader:
                optimizer.zero_grad()
                logits = model(images.to(device)).squeeze(1)
                loss = loss_function(logits, labels.to(device))
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(labels)
            scores, labels = predict(model, validation_loader)
            ap = average_precision_score(labels, scores)
            history_rows.append({
                "split_seed": split_seed, "backbone_rate": backbone_rate,
                "epoch": epoch, "train_loss": total_loss / len(fit_indices), "validation_ap": ap,
            })
            print(f"split={split_seed}, lr={backbone_rate:g}, epoch={epoch}, validation AP={ap:.4f}")
            if ap > best_ap:
                best_ap, best_epoch = ap, epoch
                best_scores = scores.copy()
                torch.save(model.state_dict(), checkpoint)
        threshold = choose_threshold(labels, best_scores)
        decisions = best_scores >= threshold
        tn, fp, fn, tp = confusion_matrix(labels, decisions, labels=[0, 1]).ravel()
        run_rows.append({
            "split_seed": split_seed, "backbone_rate": backbone_rate,
            "best_epoch": best_epoch, "validation_ap": best_ap,
            "threshold": threshold, "recall": recall_score(labels, decisions),
            "false_alarms": fp, "missed_defects": fn, "checkpoint": str(checkpoint),
        })
        del optimizer, model

runs = pd.DataFrame(run_rows)
history = pd.DataFrame(history_rows)
display(runs.drop(columns="checkpoint"))
mean_ap = runs.groupby("backbone_rate")["validation_ap"].mean()
display(mean_ap.to_frame("Mean validation AP"))
best_rate = mean_ap.idxmax()
selected = runs[(runs["backbone_rate"] == best_rate) & (runs["split_seed"] == SPLIT_SEEDS[0])].iloc[0]
THRESHOLD = float(selected["threshold"])
model = build_model()
model.load_state_dict(torch.load(selected["checkpoint"], map_location=device, weights_only=True))
print("Selected run")
display(selected)
runs.to_csv(OUTPUT_DIR / "validation_runs.csv", index=False)
history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)


## 5. Evaluate the selected model
Use the shared threshold selected on the first validation split. Compare test misses and false alarms with the previous baseline of 46 misses and 783 false alarms.

In [ ]:
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
probabilities, actual = predict(model, test_loader)
predicted = (probabilities >= THRESHOLD).astype(int)
assert len(set(actual)) == 2, "Both classes are needed for ROC AUC."
metrics = {
    "accuracy": accuracy_score(actual, predicted),
    "average_precision": average_precision_score(actual, probabilities),
    "roc_auc": roc_auc_score(actual, probabilities),
    "precision": precision_score(actual, predicted, zero_division=0),
    "recall": recall_score(actual, predicted, zero_division=0),
    "f1": f1_score(actual, predicted, zero_division=0),
}
display(pd.DataFrame([metrics], index=["ResNet-50"]).round(4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(actual, predicted, labels=[0, 1], display_labels=["Normal", "Defective"], ax=axes[0], colorbar=False)
PrecisionRecallDisplay.from_predictions(actual, probabilities, ax=axes[1])
RocCurveDisplay.from_predictions(actual, probabilities, ax=axes[2])
plt.tight_layout()
plt.show()

predictions = pd.DataFrame({
    "image": [row["image"] for row in test_data.examples],
    "product": [row["object"] for row in test_data.examples],
    "label": actual, "probability": probabilities, "prediction": predicted,
})
for title, mistakes in [
    ("Missed defects", predictions[(predictions["label"] == 1) & (predictions["prediction"] == 0)]),
    ("False alarms", predictions[(predictions["label"] == 0) & (predictions["prediction"] == 1)]),
]:
    print(title, len(mistakes))
    for row in mistakes.head(3).itertuples():
        with Image.open(DATA_DIR / row.image) as image:
            plt.figure(figsize=(3, 3))
            plt.imshow(image)
            plt.title(f"Defect probability {row.probability:.3f}")
            plt.axis("off")
            plt.show()


## 6. Save results by product
These files use the new fine-tuning results folder.

In [ ]:
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
selected.to_frame("value").to_csv(OUTPUT_DIR / "selected_run.csv")
product_rows = []
for product_name, group in predictions.groupby("product"):
    tn, fp, fn, tp = confusion_matrix(group["label"], group["prediction"], labels=[0, 1]).ravel()
    product_rows.append({
        "product": product_name, "images": len(group),
        "detected_defects": tp, "missed_defects": fn, "false_alarms": fp,
        "accuracy": accuracy_score(group["label"], group["prediction"]),
        "recall": recall_score(group["label"], group["prediction"], zero_division=0),
        "precision": precision_score(group["label"], group["prediction"], zero_division=0),
        "average_precision": average_precision_score(group["label"], group["probability"]),
        "roc_auc": roc_auc_score(group["label"], group["probability"]),
    })
product_metrics = pd.DataFrame(product_rows)
display(product_metrics.round(4))
product_metrics.to_csv(OUTPUT_DIR / "per_product_metrics.csv", index=False)
print("Saved results to", OUTPUT_DIR.resolve())
